### 🏗️ Notebook: Load Bronze Layer (Volumes -> Delta Tables)
---
#### 🎯 Purpose:
* **Ingestion:** Read raw CSV and JSON files from **Unity Catalog Volumes**.
* **Source Separation:** Maintain separate tables for each source system to ensure a **Pure Bronze** layer.
* **Auditability:** Add Metadata (ingestion timestamp & source file path) to every record.
* **Storage:** Save the data in **Delta format** for optimized performance and ACID compliance.

---
#### 📂 Data Mapping:
* **Source CSV (7 Files)** --> `restaurant_project.bronze.raw_sales_csv`
* **Source JSON (2 Files)** --> `restaurant_project.bronze.raw_sales_json`



In [0]:
import time
from pyspark.sql.functions import current_timestamp, col

In [0]:
# 1. Source Paths (Volumes)
csv_path = "/Volumes/restaurant_project/bronze/source_systems/source_csv/"
json_path = "/Volumes/restaurant_project/bronze/source_systems/source_json/"

# 2. Target Tables
target_csv_table = "restaurant_project.bronze.raw_sales_csv"
target_json_table = "restaurant_project.bronze.raw_sales_json"

def load_source(path, format_type, target_table):
    
    start_time = time.time()
    print(f">> Starting Load for {format_type.upper()} from {path}")

    #read logic
    df = spark.read.format(format_type)\
        .option("header", "true" if format_type == "csv" else "false")\
        .option("inferSchema", "true")\
        .load(path)\
        .withColumn("ingestion_timestamp", current_timestamp())\
        .withColumn("source_file", col("_metadata.file_path"))

    #write logic
    df.write.format("delta").mode("overwrite").saveAsTable(target_table)

    duration = round(time.time() - start_time, 2)
    print(f">> SUCCESS:") 
    print(f">> Row Loaded: {df.count()}")
    print(f">> Target Table: {target_table}")
    print(f'>> Load Duration: {duration} seconds')        
    print("-" * 20)

# execution logic
def load_bronze():
    batch_start = time.time()
    print('================================================')
    print('Starting Bronze Layer Load Operations')
    print('================================================')

    try:
        # load csv
        load_source(csv_path, "csv", target_csv_table)

        # load json
        load_source(json_path, "json", target_json_table)

        total_time = round(time.time() - batch_start, 2)
        print('==========================================')
        print('Loading Bronze Layer is Completed')
        print(f'   - Total Load Duration: {total_time} seconds')
        print('==========================================')

    except Exception as e:
        print('==========================================')
        print('ERROR OCCURED DURING LOADING BRONZE LAYER')
        print(f'Error Message: {str(e)}')
        print('==========================================')

# start the prosess
load_bronze()